# `mutate` — Reference

`mutate` creates or overwrites columns from a `qry()`-style spec string, each entry evaluated in order via pandas `eval()` — a plain formula per column, no lambda required.

| Syntax | Meaning |
|---|---|
| `"new_col: expr"` | one derived column |
| `"a: expr1, b: expr2"` | several in one call (comma-separated) |
| `"'new_col': expr"` | quoting the key is optional, same as `qry()` |
| `"new_col: if_else(cond, true_val, false_val)"` | dplyr-style two-branch conditional |
| `"new_col: case_when((cond1, v1), (cond2, v2), default)"` | multi-branch conditional; trailing bare value is the catch-all |
| `"new_col: map(col, {k: v, ...}, default)"` | recode a column through a lookup |

Column names *inside* the expression must stay unquoted — see the example below.

---

In [51]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import numpy as np
import pytae as pt

penguins = pt.sample_data['penguins']

## A single derived column

In [52]:
# Body mass index style ratio — a plain arithmetic formula, no lambda needed
(penguins
 .pt.mutate('bmi: body_mass_g / bill_length_mm ** 2')
 .pt.select('species', 'body_mass_g', 'bill_length_mm', 'bmi')
 .sample(10)
)

,species,body_mass_g,bill_length_mm,bmi
183,Chinstrap,4300.0,54.2,1.463760
60,Adelie,3150.0,35.7,2.471577
25,Adelie,3800.0,35.3,3.049539
244,Gentoo,5000.0,42.9,2.716786
279,Gentoo,5550.0,50.4,2.184902
47,Adelie,2975.0,37.5,2.115556
337,Gentoo,6000.0,48.8,2.519484
326,Gentoo,4700.0,41.7,2.702874
182,Chinstrap,3200.0,40.9,1.912949
90,Adelie,3550.0,35.7,2.785428


In [53]:
# Column names inside the expression must stay unquoted — quoting one turns it
# into a string literal, not a column reference, and breaks the arithmetic
try:
    penguins.pt.mutate("bmi: 'body_mass_g' / 'bill_length_mm' ** 2")
except TypeError as e:
    print('TypeError:', e)

TypeError: unsupported operand type(s) for ** or pow(): 'str' and 'int'


## Multiple entries in one call, and chaining a later entry off an earlier one
Entries are applied left to right, so a later expression can reference a column derived earlier in the *same* `mutate()` call.

In [54]:
# Two independent derived columns in one call
(penguins
 .pt.mutate('heavy: body_mass_g > 4000, mass_kg: body_mass_g / 1000')
 .pt.select('species', 'body_mass_g', 'mass_kg', 'heavy')
 .sample(10)
)

,species,body_mass_g,mass_kg,heavy
311,Gentoo,5400.0,5.40,True
314,Gentoo,4850.0,4.85,True
306,Gentoo,4600.0,4.60,True
70,Adelie,3600.0,3.60,False
192,Chinstrap,3950.0,3.95,False
319,Gentoo,5250.0,5.25,True
296,Gentoo,4600.0,4.60,True
71,Adelie,3900.0,3.90,False
195,Chinstrap,3500.0,3.50,False
116,Adelie,2900.0,2.90,False


In [55]:
# mass_lb references mass_kg, derived by the entry just before it
(penguins
 .pt.mutate('mass_kg: body_mass_g / 1000, mass_lb: mass_kg * 2.20462')
 .pt.select('species', 'mass_kg', 'mass_lb')
 .sample(10)
)

,species,mass_kg,mass_lb
227,Gentoo,5.20,11.464024
111,Adelie,4.60,10.141252
188,Chinstrap,3.85,8.487787
139,Adelie,4.25,9.369635
87,Adelie,3.50,7.716170
213,Chinstrap,3.65,8.046863
15,Adelie,3.70,8.157094
63,Adelie,4.05,8.928711
332,Gentoo,4.65,10.251483
256,Gentoo,4.95,10.912869


## String comparisons, optional key quoting, and local variables
Quoting the key (`'is_adelie'` vs `is_adelie`) is optional, same as `qry()`. String literals *inside* the expression (e.g. `'Adelie'`) still need real quotes — only the column names must stay bare. A variable from the calling scope can be referenced with an `@` prefix, same as pandas' own `eval()`/`query()`.

In [56]:
unquoted = penguins.pt.mutate("is_adelie: species == 'Adelie'")
quoted = penguins.pt.mutate("'is_adelie': species == 'Adelie'")
(unquoted['is_adelie'] == quoted['is_adelie']).all()

np.True_

In [57]:
# @-prefixed names resolve against the scope that called mutate(), not mutate()'s own internals
threshold = 4000
(penguins
 .pt.mutate('heavy: body_mass_g >= @threshold')
 .pt.select('species', 'body_mass_g', 'heavy')
 .sample(10)
)

,species,body_mass_g,heavy
87,Adelie,3500.0,False
178,Chinstrap,3400.0,False
27,Adelie,3200.0,False
139,Adelie,4250.0,True
138,Adelie,3400.0,False
11,Adelie,3700.0,False
103,Adelie,4250.0,True
263,Gentoo,4750.0,True
154,Chinstrap,3650.0,False
342,Gentoo,5200.0,True


## Overwriting an existing column

In [58]:
# mutate() can overwrite a column in place, e.g. converting units
(penguins
 .pt.mutate('body_mass_g: body_mass_g / 1000')
 .pt.select('species', 'body_mass_g')
 .sample(10)
)

,species,body_mass_g
162,Chinstrap,3.800
59,Adelie,3.750
207,Chinstrap,3.450
14,Adelie,4.400
324,Gentoo,4.725
56,Adelie,3.550
238,Gentoo,4.800
325,Gentoo,5.500
171,Chinstrap,4.400
200,Chinstrap,3.250


## Column names with spaces — backtick quoting
`pandas.eval()` uses **backticks**, not the single/double quotes used elsewhere in pytae, to reference a column name containing a space.

In [59]:
import pandas as pd
spaced = pd.DataFrame({'body mass g': [3000.0, 4000.0], 'bill length mm': [30.0, 40.0]})
spaced.pt.mutate('bmi: `body mass g` / `bill length mm` ** 2')

,body mass g,bill length mm,bmi
0,3000.0,30.0,3.333333
1,4000.0,40.0,2.500000


## Real-world pipeline: mutate → filter → select
`mutate()` chains like any other pytae method — filter on a column you just derived with `qry()`.

In [60]:
(penguins
 .pt.mutate('bmi: body_mass_g / bill_length_mm ** 2')
 .pt.qry({'bmi': ('>', 2)})
 .pt.select('species', 'bmi')
 .sample(10)
)

,species,bmi
7,Adelie,3.042352
130,Adelie,2.243211
320,Gentoo,2.061856
307,Gentoo,2.013915
50,Adelie,2.231915
108,Adelie,2.187227
268,Gentoo,2.529749
103,Adelie,2.974441
102,Adelie,2.163527
305,Gentoo,2.170004


## Conditional column creation — `if_else()`, `case_when()`, `map()`
Plain `eval()` has no ternary/`where()` support, so `mutate()` provides three dplyr-style helpers as ordinary function calls, evaluated via `np.where()`/`np.select()`/`Series.map()`: `if_else(condition, true_value, false_value)`, `case_when((cond1, val1), (cond2, val2), ..., default)`, and `map(column, {key: value, ...}, default)`. A trailing bare argument to `case_when` is the catch-all default (like SQL ELSE). Because they are ordinary calls, they compose and chain with each other and with any pandas method. String outcomes need quotes; conditions are vectorized — prefer `and`/`or`/`not` (the bitwise `&`/`|`/`~` also work).

In [61]:
# if_else(condition, true_value, false_value) — like dplyr's if_else()
(penguins
 .pt.mutate("weight_class: if_else(body_mass_g > 4000, 'heavy', 'light')")
 .pt.select('species', 'body_mass_g', 'weight_class')
 .sample(10)
)

,species,body_mass_g,weight_class
221,Gentoo,5700.0,heavy
145,Adelie,3650.0,light
94,Adelie,3300.0,light
191,Chinstrap,4500.0,heavy
112,Adelie,3200.0,light
78,Adelie,3550.0,light
102,Adelie,3075.0,light
268,Gentoo,5100.0,heavy
37,Adelie,3550.0,light
307,Gentoo,5300.0,heavy


In [62]:
# case_when((cond1, val1), (cond2, val2), ..., default) — trailing bare value is the catch-all
# checked in order, first match wins; the default must be listed last
(penguins
 .pt.mutate("size_class: case_when((body_mass_g >= 4500, 'large'), (body_mass_g >= 3500, 'medium'), 'small')")
 .pt.select('species', 'body_mass_g', 'size_class')
 .sample(10)
)

,species,body_mass_g,size_class
159,Chinstrap,3750.0,medium
97,Adelie,4350.0,medium
164,Chinstrap,3700.0,medium
55,Adelie,3700.0,medium
309,Gentoo,5550.0,large
194,Chinstrap,3550.0,medium
265,Gentoo,4900.0,large
297,Gentoo,6000.0,large
288,Gentoo,4700.0,large
81,Adelie,4700.0,large


In [63]:
# map(column, {key: value, ...}, default) — recode a column through a lookup
# unmatched keys become the default (NaN if omitted); composes like any Series
(penguins
 .pt.mutate("island_code: map(island, {'Torgersen': 'TOR', 'Biscoe': 'BIS'}, 'OTH')")
 .pt.select('species', 'island', 'island_code')
 .sample(10)
)

,species,island,island_code
231,Gentoo,Biscoe,BIS
284,Gentoo,Biscoe,BIS
11,Adelie,Torgersen,TOR
129,Adelie,Torgersen,TOR
253,Gentoo,Biscoe,BIS
48,Adelie,Dream,OTH
123,Adelie,Torgersen,TOR
189,Chinstrap,Dream,OTH
165,Chinstrap,Dream,OTH
29,Adelie,Biscoe,BIS
